# Baseline Model Development

This notebook trains and compares several baseline classification models for obesity-risk prediction.

## Objectives

- Recreate the stratified training, validation, and test datasets
- Use the reusable preprocessing module
- Establish a simple benchmark
- Train multiple classification algorithms
- Evaluate models using consistent metrics
- Select promising models for hyperparameter tuning
- Keep the test dataset untouched until final model selection

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

In [2]:
CURRENT_PATH = Path.cwd()

possible_roots = [
    CURRENT_PATH,
    *CURRENT_PATH.parents,
]

PROJECT_ROOT = next(
    (
        path
        for path in possible_roots
        if (path / "src" / "preprocessing.py").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

project_root_string = str(PROJECT_ROOT)

if project_root_string not in sys.path:
    sys.path.insert(0, project_root_string)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "obesity.csv"
)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())

Project root: c:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System
Dataset exists: True


In [3]:
from src.preprocessing import (
    PREDICTIVE_FEATURES,
    build_preprocessor,
)

print(
    "Configured predictive features:",
    len(PREDICTIVE_FEATURES),
)

test_preprocessor = build_preprocessor()

print(
    "Preprocessor type:",
    type(test_preprocessor).__name__,
)

Configured predictive features: 16
Preprocessor type: ColumnTransformer


In [4]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (20758, 18)


,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [5]:
IDENTIFIER_COLUMN = "id"
TARGET_COLUMN = "NObeyesdad"
RANDOM_STATE = 42

In [6]:
X = df[PREDICTIVE_FEATURES].copy()
y = df[TARGET_COLUMN].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (20758, 16)
Target shape: (20758,)


In [7]:
missing_features = (
    set(PREDICTIVE_FEATURES)
    - set(df.columns)
)

unexpected_features = (
    set(X.columns)
    - set(PREDICTIVE_FEATURES)
)

print("Missing features:", missing_features)
print("Unexpected features:", unexpected_features)

Missing features: set()
Unexpected features: set()


In [8]:
assert not missing_features
assert not unexpected_features
assert list(X.columns) == PREDICTIVE_FEATURES

print("Feature configuration validation passed")

Feature configuration validation passed


In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_validation, X_test, y_validation, y_test = (
    train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=y_temp,
    )
)

In [10]:
split_summary = pd.DataFrame(
    {
        "Dataset": [
            "Training",
            "Validation",
            "Test",
        ],
        "Records": [
            len(X_train),
            len(X_validation),
            len(X_test),
        ],
        "Percentage": [
            len(X_train) / len(X) * 100,
            len(X_validation) / len(X) * 100,
            len(X_test) / len(X) * 100,
        ],
    }
)

split_summary["Percentage"] = (
    split_summary["Percentage"].round(2)
)

split_summary

,Dataset,Records,Percentage
0,Training,14530,70.0
1,Validation,3114,15.0
2,Test,3114,15.0


In [11]:
train_validation_overlap = set(
    X_train.index
) & set(X_validation.index)

train_test_overlap = set(
    X_train.index
) & set(X_test.index)

validation_test_overlap = set(
    X_validation.index
) & set(X_test.index)

print(
    "Training-validation overlap:",
    len(train_validation_overlap),
)

print(
    "Training-test overlap:",
    len(train_test_overlap),
)

print(
    "Validation-test overlap:",
    len(validation_test_overlap),
)

Training-validation overlap: 0
Training-test overlap: 0
Validation-test overlap: 0


In [12]:
assert X_train.shape == (14530, 16)
assert X_validation.shape == (3114, 16)
assert X_test.shape == (3114, 16)

assert len(X_train) == len(y_train)
assert len(X_validation) == len(y_validation)
assert len(X_test) == len(y_test)

assert not train_validation_overlap
assert not train_test_overlap
assert not validation_test_overlap

assert (
    len(X_train)
    + len(X_validation)
    + len(X_test)
    == len(X)
)

print(
    "Baseline modelling dataset preparation passed"
)

Baseline modelling dataset preparation passed


In [13]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

In [14]:
CLASS_ORDER = [
    "Insufficient_Weight",
    "Normal_Weight",
    "Overweight_Level_I",
    "Overweight_Level_II",
    "Obesity_Type_I",
    "Obesity_Type_II",
    "Obesity_Type_III",
]

In [15]:
most_frequent_training_class = (
    y_train
    .value_counts()
    .idxmax()
)

most_frequent_class_count = (
    y_train
    .value_counts()
    .max()
)

most_frequent_class_percentage = (
    most_frequent_class_count
    / len(y_train)
    * 100
)

print(
    "Most frequent training class:",
    most_frequent_training_class,
)

print(
    "Training percentage:",
    f"{most_frequent_class_percentage:.2f}%",
)

Most frequent training class: Obesity_Type_III
Training percentage: 19.49%


In [16]:
dummy_classifier = DummyClassifier(
    strategy="most_frequent"
)

dummy_classifier

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None


In [17]:
dummy_classifier.fit(
    X_train,
    y_train,
)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](7,)","[0.12,0.15,0.14,...,0.19,0.12,0.12]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[object](7,)","['Insufficient_Weight','Normal_Weight','Obesity_Type_I',..., 'Obesity_Type_III','Overweight_Level_I','Overweight_Level_II']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](16,)","['Age','Height','Weight',...,'SMOKE','SCC','MTRANS']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,7
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,16
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [18]:
dummy_validation_predictions = (
    dummy_classifier.predict(
        X_validation
    )
)

In [19]:
dummy_prediction_counts = (
    pd.Series(
        dummy_validation_predictions,
        name="Predicted Class",
    )
    .value_counts()
)

dummy_prediction_counts

Predicted Class
Obesity_Type_III    3114
Name: count, dtype: int64

In [20]:
print(
    "Unique predicted classes:",
    np.unique(
        dummy_validation_predictions
    ),
)

print(
    "Number of unique predictions:",
    len(
        np.unique(
            dummy_validation_predictions
        )
    ),
)

Unique predicted classes: ['Obesity_Type_III']
Number of unique predictions: 1


In [21]:
def calculate_classification_metrics(
    model_name,
    y_true,
    y_pred,
):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "Balanced Accuracy": (
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "Weighted F1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }

In [22]:
dummy_result = calculate_classification_metrics(
    model_name="Dummy Classifier",
    y_true=y_validation,
    y_pred=dummy_validation_predictions,
)

model_results = [
    dummy_result
]

model_comparison_df = (
    pd.DataFrame(model_results)
    .set_index("Model")
    .round(4)
)

model_comparison_df

,Accuracy,Balanced Accuracy,Macro F1,Weighted F1
Model,,,,
Dummy Classifier,0.1949,0.1429,0.0466,0.0636


In [23]:
dummy_classification_report = (
    classification_report(
        y_validation,
        dummy_validation_predictions,
        labels=CLASS_ORDER,
        output_dict=True,
        zero_division=0,
    )
)

dummy_report_df = (
    pd.DataFrame(
        dummy_classification_report
    )
    .transpose()
    .round(4)
)

dummy_report_df

,precision,recall,f1-score,support
Insufficient_Weight,0.0000,0.0000,0.0000,379.0000
Normal_Weight,0.0000,0.0000,0.0000,463.0000
Overweight_Level_I,0.0000,0.0000,0.0000,364.0000
Overweight_Level_II,0.0000,0.0000,0.0000,378.0000
Obesity_Type_I,0.0000,0.0000,0.0000,436.0000
Obesity_Type_II,0.0000,0.0000,0.0000,487.0000
Obesity_Type_III,0.1949,1.0000,0.3263,607.0000
accuracy,0.1949,0.1949,0.1949,0.1949
macro avg,0.0278,0.1429,0.0466,3114.0000
weighted avg,0.0380,0.1949,0.0636,3114.0000


In [24]:
dummy_confusion_values = confusion_matrix(
    y_validation,
    dummy_validation_predictions,
    labels=CLASS_ORDER,
)

dummy_confusion_df = pd.DataFrame(
    dummy_confusion_values,
    index=[
        f"Actual: {class_name}"
        for class_name in CLASS_ORDER
    ],
    columns=[
        f"Predicted: {class_name}"
        for class_name in CLASS_ORDER
    ],
)

dummy_confusion_df

,Predicted: Insufficient_Weight,Predicted: Normal_Weight,Predicted: Overweight_Level_I,Predicted: Overweight_Level_II,Predicted: Obesity_Type_I,Predicted: Obesity_Type_II,Predicted: Obesity_Type_III
Actual: Insufficient_Weight,0,0,0,0,0,0,379
Actual: Normal_Weight,0,0,0,0,0,0,463
Actual: Overweight_Level_I,0,0,0,0,0,0,364
Actual: Overweight_Level_II,0,0,0,0,0,0,378
Actual: Obesity_Type_I,0,0,0,0,0,0,436
Actual: Obesity_Type_II,0,0,0,0,0,0,487
Actual: Obesity_Type_III,0,0,0,0,0,0,607


In [25]:
unique_dummy_predictions = np.unique(
    dummy_validation_predictions
)

assert len(dummy_validation_predictions) == len(
    y_validation
), "Prediction count does not match validation records."

assert len(unique_dummy_predictions) == 1, (
    "The most-frequent dummy classifier "
    "should predict only one class."
)

assert unique_dummy_predictions[0] == (
    most_frequent_training_class
), "Dummy prediction differs from the training majority class."

assert dummy_confusion_values.sum() == len(
    y_validation
), "Confusion matrix does not contain every validation record."

assert 0 <= dummy_result["Accuracy"] <= 1
assert 0 <= dummy_result["Balanced Accuracy"] <= 1
assert 0 <= dummy_result["Macro F1"] <= 1
assert 0 <= dummy_result["Weighted F1"] <= 1

print("Dummy classifier validation passed")

Dummy classifier validation passed
